# Lezione 5 · Le CNN, l'arte dello sguardo

La lente che scorre sulla carta, i pesi e i conti di uno strato, il riassunto, la vedetta leggera e le luci false dei naufragatori.

Ogni numero calcolato qui viene confrontato con quello scritto nel libro: un segno ✓ vuol dire che coincidono.

In [1]:
import math
import numpy as np

def controlla(nome, valore, atteso, tolleranza=0.01):
    """Confronta un valore calcolato con quello scritto nel libro."""
    ok = abs(valore - atteso) <= tolleranza
    segno = "✓" if ok else "✗ DIVERSO DAL LIBRO"
    print(f"{segno}  {nome}: calcolato {valore:.4g}, nel libro {atteso:g}")
    assert ok, nome

def conta(nome, valore, atteso):
    """Come controlla, per i numeri interi: devono coincidere esattamente."""
    ok = valore == atteso
    segno = "✓" if ok else "✗ DIVERSO DAL LIBRO"
    intero = lambda x: f"{x:,.0f}".replace(",", ".")
    print(f"{segno}  {nome}: calcolato {intero(valore)}, nel libro {intero(atteso)}")
    assert ok, nome

def sigmoide(z):
    return 1 / (1 + math.exp(-z))

def softmax(z):
    e = np.exp(np.array(z, dtype=float) - np.max(z))
    return e / e.sum()

## La convoluzione, per nome

*Nel racconto: «un piccolo riquadro alla volta», «la stessa lente dappertutto».*

Il filtro del racconto, (−1; 0; 1), toglie il punto a sinistra e aggiunge quello a destra.

In [2]:
filtro = np.array([-1, 0, 1])
for riga, atteso in [((3, 3, 3), 0), ((3, 3, 9), 6), ((3, 3, 4), 1), ((9, 3, 3), -6)]:
    controlla(f"riga {riga}", float(filtro @ riga), atteso)

def scorri(riga, filtro):
    k = len(filtro)
    return [float(filtro @ riga[i:i + k]) for i in range(len(riga) - k + 1)]

mappa = scorri(np.array([3, 3, 3, 9, 9, 9]), filtro)
print("riga di sei punti (3; 3; 3; 9; 9; 9) →", mappa)
assert mappa == [0, 6, 6, 0]

# equivarianza: se il bordo si sposta di un punto, la mappa si sposta con lui
assert scorri(np.array([3, 3, 3, 3, 9, 9, 9]), filtro) == [0, 0, 6, 6, 0]

✓  riga (3, 3, 3): calcolato 0, nel libro 0
✓  riga (3, 3, 9): calcolato 6, nel libro 6
✓  riga (3, 3, 4): calcolato 1, nel libro 1
✓  riga (9, 3, 3): calcolato -6, nel libro -6
riga di sei punti (3; 3; 3; 9; 9; 9) → [0.0, 6.0, 6.0, 0.0]


La stessa cosa in due dimensioni: una piccola carta con una crocetta, lo scoglio, e una lente 3 × 3 che cerca proprio quel disegno. Poi il filtro di Sobel su un bordo verticale.

In [3]:
def convoluzione(immagine, lente):
    k = lente.shape[0]
    n = immagine.shape[0] - k + 1
    return np.array([[np.sum(immagine[i:i + k, j:j + k] * lente) for j in range(n)] for i in range(n)])

carta = np.zeros((6, 6))
carta[3, 2:5] = 1; carta[2:5, 3] = 1              # la crocetta, con il centro in (3, 3)
lente = np.array([[0, 1, 0], [1, 1, 1], [0, 1, 0]])
mappa = convoluzione(carta, lente)
print(mappa)
dove = np.unravel_index(np.argmax(mappa), mappa.shape)
print("la lente si accende di più nel riquadro", tuple(int(i) for i in dove), "della mappa: lì sotto c'è lo scoglio")
assert tuple(dove) == (2, 2) and mappa.max() == 5

sobel = np.array([[-1, 0, 1], [-2, 0, 2], [-1, 0, 1]])
bordo = np.tile([3, 3, 3, 9, 9, 9], (6, 1))
print("Sobel su un bordo verticale, una riga della mappa:", convoluzione(bordo, sobel)[0])

[[0. 0. 1. 0.]
 [0. 2. 2. 2.]
 [1. 2. 5. 2.]
 [0. 2. 2. 2.]]
la lente si accende di più nel riquadro (2, 2) della mappa: lì sotto c'è lo scoglio
Sobel su un bordo verticale, una riga della mappa: [ 0 24 24  0]


## Dimensioni delle mappe: padding e stride

*Nel racconto: «la lente si sposta di un passo».*

In [4]:
lato = lambda n, k, p, s: (n + 2 * p - k) // s + 1
conta("n = 224, k = 3, p = 1, s = 1", lato(224, 3, 1, 1), 224)
conta("con s = 2", lato(224, 3, 1, 2), 112)

✓  n = 224, k = 3, p = 1, s = 1: calcolato 224, nel libro 224
✓  con s = 2: calcolato 112, nel libro 112


## Pesi e conti di uno strato

*Nel racconto: «risparmiano pesi, non conti».*

In [5]:
pesi = lambda k, c_in, c_out: k * k * c_in * c_out + c_out
moltiplicazioni = lambda H, W, k, c_in, c_out: H * W * k * k * c_in * c_out

conta("128 filtri 3 × 3 su 64 canali: pesi", pesi(3, 64, 128), 73856)
controlla("   su una mappa 56 × 56: moltiplicazioni (milioni)", moltiplicazioni(56, 56, 3, 64, 128) / 1e6, 231, 0.5)

# i numeri del racconto: una foto piccola, 224 × 224 a tre colori
numeri = 224 * 224 * 3
conta("numeri di una foto piccola («circa 150 mila»)", numeri, 150528)
controlla("cento neuroni completamente connessi: pesi (milioni)", 100 * numeri / 1e6, 15, 0.1)
conta("punti della foto («circa cinquantamila»)", 224 * 224, 50176)
conti_lenti = moltiplicazioni(224, 224, 3, 3, 64)
controlla("primo strato, 64 lenti: moltiplicazioni (milioni, «quasi 90»)", conti_lenti / 1e6, 86.7, 0.05)
conti_densi = 64 * numeri
controlla("64 neuroni completamente connessi: pesi (milioni, «quasi 10»)", conti_densi / 1e6, 9.6, 0.05)
assert pesi(3, 3, 64) < 2000                       # «meno di duemila pesi»
controlla("quante volte più conti fanno le lenti («quasi nove»)", conti_lenti / conti_densi, 9, 0.05)

✓  128 filtri 3 × 3 su 64 canali: pesi: calcolato 73.856, nel libro 73.856
✓     su una mappa 56 × 56: moltiplicazioni (milioni): calcolato 231.2, nel libro 231
✓  numeri di una foto piccola («circa 150 mila»): calcolato 150.528, nel libro 150.528
✓  cento neuroni completamente connessi: pesi (milioni): calcolato 15.05, nel libro 15
✓  punti della foto («circa cinquantamila»): calcolato 50.176, nel libro 50.176
✓  primo strato, 64 lenti: moltiplicazioni (milioni, «quasi 90»): calcolato 86.7, nel libro 86.7
✓  64 neuroni completamente connessi: pesi (milioni, «quasi 10»): calcolato 9.634, nel libro 9.6
✓  quante volte più conti fanno le lenti («quasi nove»): calcolato 9, nel libro 9


## Il riassunto e il campo recettivo

*Nel racconto: «la carta d'insieme», «ogni lente vede una porzione sempre più ampia».*

In [6]:
r, j = 1, 1
attesi = [3, 4, 8, 10, 18]
for (nome, k, s), atteso in zip([("convoluzione 3 × 3", 3, 1), ("pooling 2 × 2", 2, 2), ("convoluzione 3 × 3", 3, 1),
                                 ("pooling 2 × 2", 2, 2), ("convoluzione 3 × 3", 3, 1)], attesi):
    r, j = r + (k - 1) * j, j * s
    conta(f"dopo {nome}: campo recettivo", r, atteso)

✓  dopo convoluzione 3 × 3: campo recettivo: calcolato 3, nel libro 3
✓  dopo pooling 2 × 2: campo recettivo: calcolato 4, nel libro 4
✓  dopo convoluzione 3 × 3: campo recettivo: calcolato 8, nel libro 8
✓  dopo pooling 2 × 2: campo recettivo: calcolato 10, nel libro 10
✓  dopo convoluzione 3 × 3: campo recettivo: calcolato 18, nel libro 18


## Passi, epoche e il 2012

*Nel racconto: il libro dei kata; «la rete che ha cambiato la storia della visione».*

In [7]:
conta("il diario di duecento serate, a gruppi di venti: passi per epoca", 200 // 20, 10)
passi_epoca = 1_200_000 / 128
controlla("AlexNet: passi per epoca", passi_epoca, 9400, 30)
controlla("AlexNet: passi in 90 epoche (migliaia)", passi_epoca * 90 / 1e3, 850, 10)

✓  il diario di duecento serate, a gruppi di venti: passi per epoca: calcolato 10, nel libro 10
✓  AlexNet: passi per epoca: calcolato 9375, nel libro 9400
✓  AlexNet: passi in 90 epoche (migliaia): calcolato 843.8, nel libro 850


## L'apprendimento trasferito

*Nel racconto: il velista sulla barca a noleggio; «tavola buona», «nodo», «crepa».*

In [8]:
conta("nuovo strato finale per tre risposte, su 512 numeri", 512 * 3 + 3, 1539)
print(f"   è lo {1539 / 11.7e6:.3%} degli 11,7 milioni di parametri della ResNet-18")

✓  nuovo strato finale per tre risposte, su 512 numeri: calcolato 1.539, nel libro 1.539
   è lo 0.013% degli 11,7 milioni di parametri della ResNet-18


## Il rilevamento di oggetti

*Nel racconto: la vedetta; il riquadro; «guardi una volta sola».*

In [9]:
controlla("tempo per immagine a 30 al secondo (millesimi)", 1000 / 30, 33, 0.5)

def iou(a, b):          # riquadri (x1, y1, x2, y2)
    larg = max(0, min(a[2], b[2]) - max(a[0], b[0]))
    alt = max(0, min(a[3], b[3]) - max(a[1], b[1]))
    inter = larg * alt
    unione = (a[2] - a[0]) * (a[3] - a[1]) + (b[2] - b[0]) * (b[3] - b[1]) - inter
    return inter, unione, inter / unione

inter, unione, valore = iou((0, 0, 4, 4), (1, 0, 5, 4))
conta("intersezione", inter, 12)
conta("unione", unione, 20)
controlla("IoU", valore, 0.6)

✓  tempo per immagine a 30 al secondo (millesimi): calcolato 33.33, nel libro 33
✓  intersezione: calcolato 12, nel libro 12
✓  unione: calcolato 20, nel libro 20
✓  IoU: calcolato 0.6, nel libro 0.6


## La vedetta leggera

*Nel racconto: le lenti più furbe, i numeri più corti, il mozzo che imita il nostromo, l'occhio fatto apposta.*

In [10]:
k, c_out = 3, 256
rapporto = 1 / c_out + 1 / k ** 2
controlla("lenti più furbe: conti rispetto a una convoluzione normale", rapporto, 0.115, 0.001)
print(f"   cioè {1 / rapporto:.1f} volte meno conti: «quasi nove» nei conti, «circa otto» nel racconto")
# lo stesso rapporto, contando le moltiplicazioni una per una (64 canali in ingresso, mappa 56 × 56)
normale = 56 * 56 * k * k * 64 * c_out
separabile = 56 * 56 * k * k * 64 + 56 * 56 * 64 * c_out
controlla("   contando le moltiplicazioni", separabile / normale, 0.115, 0.001)
controlla("numeri più corti: da 32 a 8 bit, volte meno memoria", 32 / 8, 4)

✓  lenti più furbe: conti rispetto a una convoluzione normale: calcolato 0.115, nel libro 0.115
   cioè 8.7 volte meno conti: «quasi nove» nei conti, «circa otto» nel racconto
✓     contando le moltiplicazioni: calcolato 0.115, nel libro 0.115
✓  numeri più corti: da 32 a 8 bit, volte meno memoria: calcolato 4, nel libro 4


## Gli esempi avversari

*Nel racconto: il panda diventato gibbone; le luci false dei naufragatori.*

Un neurone lineare con 150.528 numeri in ingresso: ogni pixel si sposta di ε nella direzione che fa crescere la spinta.

In [11]:
eps, peso_medio = 0.007, 0.01
controlla("di quanto cambia la spinta", eps * peso_medio * numeri, 10.5, 0.05)

# la prova: pesi a caso con valore assoluto medio 0,01, un'immagine a caso, il disturbo di FGSM
rng = np.random.default_rng(0)
w = rng.laplace(0, 0.01, numeri)                   # |w| vale in media 0,01
x = rng.random(numeri)
x_falsa = np.clip(x + eps * np.sign(w), 0, 1)      # per un neurone lineare il gradiente rispetto all'immagine è w
print(f"   ogni pixel si è mosso al più di {np.max(np.abs(x_falsa - x)):.3f}; la spinta è cambiata di {float(w @ x_falsa - w @ x):.1f}")
assert 5 < float(w @ x_falsa - w @ x) <= eps * np.abs(w).sum() + 1e-9

✓  di quanto cambia la spinta: calcolato 10.54, nel libro 10.5
   ogni pixel si è mosso al più di 0.007; la spinta è cambiata di 10.5


## Kata per casa

**Esercizio 1.** I parametri di un filtro 3 × 3 su 3 canali, di 64 filtri, e di 64 neuroni completamente connessi.

In [12]:
conta("parametri di un filtro", pesi(3, 3, 1), 28)
conta("parametri di 64 filtri", pesi(3, 3, 64), 1792)
conta("pesi di 64 neuroni completamente connessi", 64 * numeri, 9633792)
print(f"   la convoluzione usa {64 * numeri / pesi(3, 3, 64):.0f} volte meno parametri: «circa cinquemila»")

✓  parametri di un filtro: calcolato 28, nel libro 28
✓  parametri di 64 filtri: calcolato 1.792, nel libro 1.792
✓  pesi di 64 neuroni completamente connessi: calcolato 9.633.792, nel libro 9.633.792
   la convoluzione usa 5376 volte meno parametri: «circa cinquemila»


**Esercizio 2.** Il max pooling 2 × 2 della mappa 4 × 4.

In [13]:
mappa = np.array([[1, 0, 2, 3], [4, 6, 6, 8], [3, 1, 1, 0], [1, 2, 2, 4]])
riassunto = mappa.reshape(2, 2, 2, 2).max(axis=(1, 3))
print(riassunto)
assert riassunto.tolist() == [[6, 8], [3, 4]]
conta("da 16 numeri a", riassunto.size, 4)

[[6 8]
 [3 4]]
✓  da 16 numeri a: calcolato 4, nel libro 4
